## Árvores de regressão - exercícios 02

Este exercício será uma continuação do anterior, mesma base, mesmas variáveis - vamos tentar buscar a 'melhor árvore'.


*Atenção - Utilizar a base de dados em anexo que é a mesma base que utilizamos na atividade anterior! A base Boston, assim como para a primeira atividade foi descontinuada e não deve ser utilizada*

In [ ]:
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score


### 1. Execute os passos do exercício anterior, até que você tenha uma árvore de regressão predizendo o valor do imóvel na base de treinamento.

In [ ]:
# carregar dados
df = pd.read_csv("housing.csv")

# tratar dados
df = pd.get_dummies(df, columns=['ocean_proximity'], drop_first=True)
df.fillna(df.median(numeric_only=True), inplace=True)

# separar X e y
X = df.drop('median_house_value', axis=1)
y = df['median_house_value']

# treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# árvore inicial (sem limitar)
tree = DecisionTreeRegressor(random_state=42)
tree.fit(X_train, y_train)

### 2.  Calcule o caminho indicado pelos CCP-alfas dessa árvore.

In [ ]:
path = tree.cost_complexity_pruning_path(X_train, y_train)

ccp_alphas = path.ccp_alphas
impurities = path.impurities

print(ccp_alphas)

### 3. Paca cada valor de alpha obtido no item 2, treine uma árvore com o respectivo alfa, e guarde essa árvore em uma lista.

In [ ]:
trees = []

for alpha in ccp_alphas:
    model = DecisionTreeRegressor(random_state=42, ccp_alpha=alpha)
    model.fit(X_train, y_train)
    trees.append(model)

### 4. Para cada árvore na lista, calcule o MSE da árvore.

In [ ]:
train_mse = []
test_mse = []

for model in trees:
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    train_mse.append(mean_squared_error(y_train, y_train_pred))
    test_mse.append(mean_squared_error(y_test, y_test_pred))

### 5. Monte um gráfico do MSE pelo alpha, escolha um valor de alpha perto do ponto de mínimo do MSE

In [ ]:

plt.figure(figsize=(10,6))
plt.plot(ccp_alphas, train_mse, marker='o', label="Treino")
plt.plot(ccp_alphas, test_mse, marker='o', label="Teste")

plt.xlabel("Alpha")
plt.ylabel("MSE")
plt.title("MSE vs CCP Alpha")
plt.legend()
plt.show()

In [ ]:
# pegar melhor alpha
best_alpha = ccp_alphas[test_mse.index(min(test_mse))]
print("Melhor alpha:", best_alpha)

### 6. Calcule o R-quadrado dessa árvore encontrada no item acima

In [ ]:
best_tree = DecisionTreeRegressor(random_state=42, ccp_alpha=best_alpha)
best_tree.fit(X_train, y_train)

y_pred = best_tree.predict(X_test)

r2 = r2_score(y_test, y_pred)
print("R²:", r2)

### 7. Visualize esta árvore.

In [ ]:
plt.figure(figsize=(20,10))
tree.plot_tree(
    best_tree,
    feature_names=X.columns,
    filled=True
)
plt.show()